In [59]:
import urllib
import time
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd

DRIVER_PATH = 'chromedriver.exe'
BASE_URL = 'https://map.naver.com/v5/search/'


station = input()

keyword = station + "맛집"

encText = urllib.parse.quote(keyword)
url = BASE_URL + encText

# 데이터를 담을 리스트와 각 항목에 대한 접근을 확인하기 위한 cnt 변수 생성
outList = []
cnt = 1
nowPage = 1

범일역


In [114]:
# selenium 브라우저로 해당 URL 접속
browser = webdriver.Chrome(DRIVER_PATH)
time.sleep(2)
browser.get(url)

# 필요한 정보가 있는 iframe 페이지에 id명을 통한 접근
browser.switch_to.frame('searchIframe')
time.sleep(2)

for i in range(nowPage+1):
    pages = browser.find_elements_by_css_selector('div._2ky45 a')
    for page in pages:
        print(page.text)
        if page.text == str(i):
            page.click()
            time.sleep(2)
            break

# 탐색이 끝난 후 다음 페이지로 이동

# 이전 페이지, 5개의 페이지 번호, 다음 페이지 버튼으로 구성된
# 7개의 <a> 태그 중 다음 페이지 버튼인 마지막 <a> 태그 선택
nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]

# 해당 버튼의 class명 확인
isLastPage = nextBtn.get_attribute("class")
print(isLastPage)


# 스크롤을 하면 추가되는 <li> 항목을 모두 가져오기 위한 div 스크롤 조작  
container = browser.find_element_by_id('_pcmap_list_scroll_container')
browser.execute_script("arguments[0].scrollBy(0, 3000)", container)                                
time.sleep(2)
browser.execute_script("arguments[0].scrollBy(0, 4500)", container)
time.sleep(2)
browser.execute_script("arguments[0].scrollBy(0, 5000)", container)
time.sleep(2)

targets = browser.find_elements_by_css_selector("li[class='_1EKsQ _12tNp']")

# 페이지당 50개인 <li> 항목을 모두 가져왔는지 확인 
print(len(targets))
targets = targets[cnt-(nowPage-1)*50-1:]
print(len(targets))

for t in targets:

    pageDetail = t.find_element_by_css_selector('a._3nCBm')
    time.sleep(2)
    pageDetail.click()
    time.sleep(3)
    browser.switch_to.default_content()
    browser.switch_to.frame('entryIframe')

    # 스크롤 된 페이지 소스를 bs4 객체로 변환
    htmlStr = browser.page_source
    bs = BeautifulSoup(htmlStr,'html.parser')

    browser.switch_to.default_content()
    browser.switch_to.frame('searchIframe')

    targetDiv = bs.select_one("div[role='main']")

    # 썸네일 이미지
    img = targetDiv.select_one('#ibu_1')
    if img:
        img = img['style'].split('"')[1]
    else:
        img = None

    # 점포명, 분류
    titleInfo = targetDiv.select_one('#_title')
    name = titleInfo.select_one('span._3XamX').text
    category = titleInfo.select_one('span._3ocDE').text

    # 별점 (별점이 없으면 None을 저장)
    score = targetDiv.select_one("span[class='_1Y6hi _1A8_M'] em")
    if score:
        score = float(score.text)
    else:
        score = None

    reviewInfo = targetDiv.select("span[class='_1Y6hi']")
    visitNum = None
    blogNum = None
    for i in reviewInfo:
        info = i.text.split(' ')
        if info[0] in ['방문자리뷰','주문자리뷰']:
            visitNum = int(info[1].replace(',',''))
        elif '블로그리뷰' == info[0]:
            blogNum = int(info[1].replace(',',''))


    phone = targetDiv.select_one('span._3ZA0S')
    if phone:
        phone = phone.text
    else:
        phone = None

    loc = targetDiv.select_one('span._2yqUQ')
    if loc:
        loc = loc.text
    else:
        loc = None

    distance = targetDiv.select_one('div._2P6sT')
    if distance:
        distance = distance.text[1:]
    else:
        distance = None
    
    
    outList.append([category, name, score, img, visitNum, blogNum, phone, loc, distance, station])
    
    # 접근 중인 항목 번호 
    print(cnt)
    cnt += 1
    
# browser.quit()

C:\Users\pnu\AppData\Local\Temp\ipykernel_11468\767568626.py:2: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  browser = webdriver.Chrome(DRIVER_PATH)
C:\Users\pnu\AppData\Local\Temp\ipykernel_11468\767568626.py:11: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  pages = browser.find_elements_by_css_selector('div._2ky45 a')


이전페이지
1
2
3
4
5
다음페이지
이전페이지
1
이전페이지
1
2
이전페이지
1
2
3
이전페이지
1
2
3
4


C:\Users\pnu\AppData\Local\Temp\ipykernel_11468\767568626.py:23: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]
C:\Users\pnu\AppData\Local\Temp\ipykernel_11468\767568626.py:31: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  container = browser.find_element_by_id('_pcmap_list_scroll_container')


_2bgjK 
50
0


C:\Users\pnu\AppData\Local\Temp\ipykernel_11468\767568626.py:39: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  targets = browser.find_elements_by_css_selector("li[class='_1EKsQ _12tNp']")


In [108]:
print(f'''확인: 현재 cnt : {cnt}
현재 페이지 번호: {nowPage}
가져온 레코드 수: {len(outList)}

현재까지 저장된 첫째 레코드 정보: {outList[0]}
현재까지 저장된 둘째 레코드 정보: {outList[1]}
현재까지 저장된 마지막 이전 레코드 정보: {outList[-2]}
현재까지 저장된 마지막 레코드 정보: {outList[-1]}

''')

확인: 현재 cnt : 201
현재 페이지 번호: 4
가져온 레코드 수: 200

현재까지 저장된 첫째 레코드 정보: ['생선구이', '신선식당', 4.28, 'https://search.pstatic.net/common/?autoRotate=true&type=w560_sharpen&src=https%3A%2F%2Fldb-phinf.pstatic.net%2F20200807_298%2F1596790597743BMlOw_JPEG%2FFP5ROuIPRM-HCiYuT-L7wwFF.jpeg.jpg', 606, 329, '051-631-9209', '부산 동구 조방로49번길 13 제일상가아파트', '범일역 4번 출구에서144m', '범일역']
현재까지 저장된 둘째 레코드 정보: ['생선회', '감포참가자미', 4.39, 'https://search.pstatic.net/common/?autoRotate=true&type=w560_sharpen&src=https%3A%2F%2Fldb-phinf.pstatic.net%2F20190308_120%2F1552012580100OwemJ_JPEG%2Fb2N3MqreHI-WW0TSh4nWZNht.jpg', 335, 69, '0507-1309-3187', '부산 동구 범일로102번길 16-20', '범일역 2번 출구에서302m', '범일역']
현재까지 저장된 마지막 이전 레코드 정보: ['한식', '덕&돈 범일점', None, 'https://search.pstatic.net/common/?autoRotate=true&quality=95&type=w750&src=https%3A%2F%2Fldb-phinf.pstatic.net%2F20210803_10%2F162799028023983zKE_PNG%2FQKaLPP_DLLx5sTyXJQ4bw17r.PNG.png', None, 1, '051-637-5293', '부산 동구 자성공원로3번길 14', '범일역 2번 출구에서295m', '범일역']
현재까지 저장된 마지막 레코드 정보: ['카페,디저

In [95]:

# 클래스명이 "_2bgjK "이면 (공백 포함 주의 !!)
# 마지막 페이지가 아니므로 다음 페이지로 이동
if isLastPage == "_2bgjK ":
    print("this is not last page. 이게 뜨면 위에 코드 다시 실행.")
    nowPage += 1

# class명이 "_2bgjK _34lTS"이면 마지막 페이지이므로 반복문 종료
else:
    print("끝. 다 했음. 마지막 페이지임. 더 시키지 마셈.")
    # 엑셀에서 한글이 깨지지 않는 'utf-8-sig' 인코딩으로 csv 파일 추출 
    df = pd.DataFrame(outList,columns=['분류','점포명','별점','이미지','방문자 리뷰 수', '블로그 리뷰 수', '전화번호','위치', '역거리','역명'])
    df.to_csv(f"{keyword}CSV.csv",encoding='utf-8-sig')

this is not last page. 이게 뜨면 위에 코드 다시 실행.


In [115]:
df = pd.DataFrame(outList,columns=['분류','점포명','별점','이미지','방문자 리뷰 수', '블로그 리뷰 수', '전화번호','위치', '역거리','역명'])
df.to_csv(f"{keyword}CSV.csv",encoding='utf-8-sig')